# MIMIC IV Data Creation

This notebook contains scripts for processing and creating datasets from MIMIC IV.


In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
from typing import Optional

# Set base paths - all data from xiaochen
BASE_DATA_PATH = Path("/data/xiaochen/physionet.org/files/")
MIMICIV_PATH = BASE_DATA_PATH / "mimiciv"
MIMIC_CXR_JPG_PATH = BASE_DATA_PATH / "mimic-cxr-jpg"
MIMIC_CXR_PATH = BASE_DATA_PATH / "mimic-cxr"  # Original CXR with .pt and .txt files
MIMIC_IV_ECG_PATH = BASE_DATA_PATH / "mimic-iv-ecg"
MIMIC_IV_NOTE_PATH = BASE_DATA_PATH / "mimic-iv-note"

OUTPUT_PATH = Path("./output/")
OUTPUT_PATH.mkdir(exist_ok=True)

print("MIMIC IV Data Creation Script")
print(f"Base data path: {BASE_DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")


MIMIC IV Data Creation Script
Base data path: /data/xiaochen/physionet.org/files
Output path: output


## Data Loading Functions

Functions to load data from different MIMIC IV subsets.


In [5]:
def load_mimiciv_ehr(lazy: bool = False):
    """
    Load MIMIC-IV EHR data from xiaochen directory.
    
    Args:
        lazy: If True, only load essential files (admissions, patients, diagnoses)
        
    Returns:
        Tuple of dataframes: (admissions, patients, diagnoses, prescriptions, labevents, procedures)
    """
    print("Loading MIMIC-IV EHR dataset files...")
    # Note: Version 2.2 is available in xiaochen directory (not 3.0)
    path = MIMICIV_PATH / "2.2" / "hosp"
    
    print("Loading admissions...")
    admissions = pd.read_csv(path / "admissions.csv")
    print("Loading patients...")
    patients = pd.read_csv(path / "patients.csv")
    print("Loading diagnoses...")
    diagnoses = pd.read_csv(path / "diagnoses_icd.csv")
    
    # Convert column names to lowercase for consistency
    admissions.columns = admissions.columns.str.lower()
    patients.columns = patients.columns.str.lower()
    diagnoses.columns = diagnoses.columns.str.lower()
    
    if lazy:
        return admissions, patients, diagnoses, None, None, None
    
    print("Loading prescriptions...")
    prescriptions = pd.read_csv(path / "prescriptions.csv")
    prescriptions.columns = prescriptions.columns.str.lower()
    
    # Clean prescriptions
    prescriptions_cleaned = prescriptions.dropna(subset=['formulary_drug_cd', 'subject_id'])
    print(f'Dropped {prescriptions.shape[0] - prescriptions_cleaned.shape[0]} rows with missing formulary_drug_cd or subject_id.')
    
    print("Loading labevents...")
    labevents = pd.read_csv(path / "labevents.csv")
    labevents.columns = labevents.columns.str.lower()
    # Cast itemid to string
    labevents['itemid'] = labevents['itemid'].astype(str)
    
    print("Loading procedures...")
    procedures = pd.read_csv(path / "procedures_icd.csv")
    procedures.columns = procedures.columns.str.lower()
    
    print("✅ MIMIC-IV EHR data loaded successfully.")
    return admissions, patients, diagnoses, prescriptions_cleaned, labevents, procedures


In [6]:
def load_mimiciv_cxr(lazy: bool = False):
    """
    Load MIMIC-CXR dataset files from xiaochen directory.
    
    Note on data structure:
    - JPG images are in: mimic-cxr-jpg/2.0.0/files/p{first_two_digits}/p{subject_id}/s{study_id}/{dicom_id}.jpg
    - TXT notes are in: mimic-cxr/2.0.0/files/p{first_two_digits}/p{subject_id}/s{study_id}.txt
    - Each study can have multiple images (different DICOM IDs) but shares one txt file
    - We use JPG images (not the original .pt files) paired with their corresponding txt notes
    
    Args:
        lazy: If True, only load metadata
        
    Returns:
        Tuple of dataframes: (metadata, chexpert, negbio, split, None)
        Note: text_set_label is not available in this dataset version (2.0.0)
    """
    print("Loading MIMIC-CXR dataset files...")
    
    # Construct paths - version 2.0.0 (actual version in xiaochen directory)
    jpg_root = MIMIC_CXR_JPG_PATH / "2.0.0" / "files"
    metadata_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-metadata.csv"
    chexpert_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-chexpert.csv"
    negbio_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-negbio.csv"
    split_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-split.csv"
    
    print("Loading metadata...")
    metadata = pd.read_csv(metadata_path)
    
    if lazy:
        return metadata, None, None, None, None
    
    print("Loading chexpert...")
    chexpert = pd.read_csv(chexpert_path)
    
    print("Loading negbio...")
    negbio = pd.read_csv(negbio_path)
    
    print("Loading split...")
    split = pd.read_csv(split_path)
    
    print("✅ MIMIC-CXR data loaded successfully.")
    return metadata, chexpert, negbio, split, None


In [7]:
def load_mimiciv_ecg(lazy: bool = False):
    """
    Load MIMIC-IV ECG dataset files from xiaochen directory.
    
    Args:
        lazy: If True, only load machine_measurements
        
    Returns:
        Tuple of dataframes: (patients, admissions, ecg_report, ecg_record_list)
    """
    print("Loading MIMIC-IV ECG dataset files...")
    
    ecg_path = MIMIC_IV_ECG_PATH / "1.0"
    
    print("Loading ECG machine measurements...")
    ecg_report = pd.read_csv(ecg_path / "machine_measurements.csv")
    
    if lazy:
        return None, None, ecg_report, None
    
    # Load patient and admission data from MIMIC-IV for matching
    # Note: Version 2.2 is available in xiaochen directory
    mimiciv_path = MIMICIV_PATH / "2.2" / "hosp"
    print("Loading patient data from MIMIC-IV...")
    patients = pd.read_csv(mimiciv_path / "patients.csv")
    patients.columns = patients.columns.str.lower()
    
    print("Loading admissions data from MIMIC-IV...")
    admissions = pd.read_csv(mimiciv_path / "admissions.csv")
    admissions.columns = admissions.columns.str.lower()
    
    print("Loading ECG record list...")
    ecg_record_list = pd.read_csv(ecg_path / "record_list.csv")
    
    print("✅ MIMIC-IV ECG data loaded successfully.")
    return patients, admissions, ecg_report, ecg_record_list


In [8]:
def load_mimiciv_icustays():
    """
    Load MIMIC-IV ICU stays dataset from xiaochen directory.
    
    Returns:
        DataFrame: ICU stays data
    """
    print("Loading MIMIC-IV ICU stays dataset...")
    # Note: Version 2.2 is available in xiaochen directory
    path = MIMICIV_PATH / "2.2" / "icu"
    
    icustays = pd.read_csv(path / "icustays.csv")
    icustays.columns = icustays.columns.str.lower()
    
    print("✅ MIMIC-IV ICU stays data loaded successfully.")
    return icustays


In [9]:
def load_mimiciv_notes(lazy: bool = False):
    """
    Load MIMIC-IV Notes dataset files from xiaochen directory.
    
    Args:
        lazy: If True, only load main note files (discharge, radiology)
        
    Returns:
        Tuple of dataframes: (discharge, radiology, discharge_detail, radiology_detail)
    """
    print("Loading MIMIC-IV Notes dataset files...")
    path = MIMIC_IV_NOTE_PATH / "2.2" / "note"
    
    print("Loading discharge notes...")
    discharge = pd.read_csv(path / "discharge.csv")
    discharge.columns = discharge.columns.str.lower()
    
    print("Loading radiology notes...")
    radiology = pd.read_csv(path / "radiology.csv")
    radiology.columns = radiology.columns.str.lower()
    
    if lazy:
        return discharge, radiology, None, None
    
    print("Loading discharge detail...")
    discharge_detail = pd.read_csv(path / "discharge_detail.csv")
    discharge_detail.columns = discharge_detail.columns.str.lower()
    
    print("Loading radiology detail...")
    radiology_detail = pd.read_csv(path / "radiology_detail.csv")
    radiology_detail.columns = radiology_detail.columns.str.lower()
    
    print("✅ MIMIC-IV Notes data loaded successfully.")
    return discharge, radiology, discharge_detail, radiology_detail


## Test Data Loading

Test loading all data sources to verify paths are correct.


In [10]:
# Test loading all data sources (lazy mode for faster testing)
print("=" * 60)
print("Testing Data Loading Functions")
print("=" * 60)

# Load EHR data
ehr_admissions, ehr_patients, ehr_diagnoses, _, _, _ = load_mimiciv_ehr(lazy=True)
print(f"\nEHR: {len(ehr_admissions)} admissions, {len(ehr_patients)} patients, {len(ehr_diagnoses)} diagnoses")

# Load CXR data
cxr_metadata, _, _, _, _ = load_mimiciv_cxr(lazy=True)
print(f"\nCXR: {len(cxr_metadata)} records")

# Load ECG data
_, _, ecg_report, _ = load_mimiciv_ecg(lazy=True)
print(f"\nECG: {len(ecg_report)} records")

# Load ICU stays
icustays = load_mimiciv_icustays()
print(f"\nICU: {len(icustays)} ICU stays")

# Load Notes
discharge_notes, radiology_notes, _, _ = load_mimiciv_notes(lazy=True)
print(f"\nNotes: {len(discharge_notes)} discharge notes, {len(radiology_notes)} radiology notes")

print("\n" + "=" * 60)
print("✅ All data loading functions tested successfully!")
print("=" * 60)


Testing Data Loading Functions
Loading MIMIC-IV EHR dataset files...
Loading admissions...
Loading patients...
Loading diagnoses...

EHR: 431231 admissions, 299712 patients, 4756326 diagnoses
Loading MIMIC-CXR dataset files...
Loading metadata...

CXR: 377110 records
Loading MIMIC-IV ECG dataset files...
Loading ECG machine measurements...


/tmp/ipykernel_726591/1068643345.py:16: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  ecg_report = pd.read_csv(ecg_path / "machine_measurements.csv")



ECG: 800035 records
Loading MIMIC-IV ICU stays dataset...
✅ MIMIC-IV ICU stays data loaded successfully.

ICU: 73181 ICU stays
Loading MIMIC-IV Notes dataset files...
Loading discharge notes...
Loading radiology notes...

Notes: 331793 discharge notes, 2321355 radiology notes

✅ All data loading functions tested successfully!


## Detailed CXR Data Test

Comprehensive test of MIMIC-CXR data loading and inspection.


In [11]:
# Test MIMIC-CXR data loading
print("=" * 70)
print("Testing MIMIC-CXR Data Loading")
print("=" * 70)

# Test lazy loading first (metadata only)
print("\n--- Testing Lazy Loading (Metadata Only) ---")
cxr_metadata, _, _, _, _ = load_mimiciv_cxr(lazy=True)

print(f"\n✅ Metadata loaded successfully!")
print(f"   Total records: {len(cxr_metadata)}")
print(f"   Columns: {list(cxr_metadata.columns)}")
print(f"\n   First few rows:")
print(cxr_metadata.head())

print(f"\n   Data types:")
print(cxr_metadata.dtypes)

print(f"\n   Sample statistics:")
print(f"   - Unique subjects: {cxr_metadata['subject_id'].nunique() if 'subject_id' in cxr_metadata.columns else 'N/A'}")
print(f"   - Unique studies: {cxr_metadata['study_id'].nunique() if 'study_id' in cxr_metadata.columns else 'N/A'}")
print(f"   - Unique DICOM IDs: {cxr_metadata['dicom_id'].nunique() if 'dicom_id' in cxr_metadata.columns else 'N/A'}")

print("\n" + "=" * 70)


Testing MIMIC-CXR Data Loading

--- Testing Lazy Loading (Metadata Only) ---
Loading MIMIC-CXR dataset files...
Loading metadata...

✅ Metadata loaded successfully!
   Total records: 377110
   Columns: ['dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'StudyDate', 'StudyTime', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning', 'PatientOrientationCodeSequence_CodeMeaning']

   First few rows:
                                       dicom_id  subject_id  study_id  \
0  02aa804e-bde0afdd-112c0b34-7bc16630-4e384014    10000032  50414267   
1  174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962    10000032  50414267   
2  2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab    10000032  53189527   
3  e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c    10000032  53189527   
4  68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714    10000032  53911762   

  PerformedProcedureStepDescription ViewPosition  Rows  Columns  StudyDate  \
0        

In [12]:
# Test full CXR loading (all files)
print("\n--- Testing Full CXR Loading (All Files) ---")
try:
    cxr_metadata_full, cxr_chexpert, cxr_negbio, cxr_split, _ = load_mimiciv_cxr(lazy=False)
    
    print(f"\n✅ All CXR files loaded successfully!")
    print(f"\n   Metadata: {len(cxr_metadata_full)} records")
    
    if cxr_chexpert is not None:
        print(f"   CheXpert: {len(cxr_chexpert)} records")
        print(f"   CheXpert columns: {list(cxr_chexpert.columns[:5])}...")  # Show first 5 columns
    
    if cxr_negbio is not None:
        print(f"   NegBio: {len(cxr_negbio)} records")
        print(f"   NegBio columns: {list(cxr_negbio.columns[:5])}...")  # Show first 5 columns
    
    if cxr_split is not None:
        print(f"   Split: {len(cxr_split)} records")
        print(f"   Split columns: {list(cxr_split.columns)}")
        if 'split' in cxr_split.columns:
            print(f"   Split distribution:")
            print(cxr_split['split'].value_counts())
    
    print("\n   Sample metadata record:")
    if len(cxr_metadata_full) > 0:
        print(cxr_metadata_full.iloc[0].to_dict())
        
except Exception as e:
    print(f"\n❌ Error loading full CXR data: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 70)



--- Testing Full CXR Loading (All Files) ---
Loading MIMIC-CXR dataset files...
Loading metadata...
Loading chexpert...
Loading negbio...
Loading split...
✅ MIMIC-CXR data loaded successfully.

✅ All CXR files loaded successfully!

   Metadata: 377110 records
   CheXpert: 227827 records
   CheXpert columns: ['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation']...
   NegBio: 227827 records
   NegBio columns: ['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation']...
   Split: 377110 records
   Split columns: ['dicom_id', 'study_id', 'subject_id', 'split']
   Split distribution:
split
train       368960
test          5159
validate      2991
Name: count, dtype: int64

   Sample metadata record:
{'dicom_id': '02aa804e-bde0afdd-112c0b34-7bc16630-4e384014', 'subject_id': 10000032, 'study_id': 50414267, 'PerformedProcedureStepDescription': 'CHEST (PA AND LAT)', 'ViewPosition': 'PA', 'Rows': 3056, 'Columns': 2544, 'StudyDate': 21800506, 'StudyTime': 213

In [13]:
# Verify path existence and check a sample file
print("\n--- Verifying CXR File Paths ---")
jpg_root = MIMIC_CXR_JPG_PATH / "2.0.0" / "files"
metadata_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-metadata.csv"

print(f"   JPG root exists: {jpg_root.exists()}")
print(f"   Metadata file exists: {metadata_path.exists()}")

if metadata_path.exists():
    print(f"   Metadata file size: {metadata_path.stat().st_size / (1024*1024):.2f} MB")

# Check if we can find sample image files
if jpg_root.exists():
    sample_dirs = list(jpg_root.iterdir())[:3]
    print(f"\n   Sample directories in JPG root: {[d.name for d in sample_dirs]}")
    if sample_dirs:
        sample_files = list(sample_dirs[0].iterdir())[:3]
        print(f"   Sample files in first directory: {[f.name for f in sample_files]}")

print("\n" + "=" * 70)
print("✅ CXR Data Test Complete!")
print("=" * 70)



--- Verifying CXR File Paths ---
   JPG root exists: True
   Metadata file exists: True
   Metadata file size: 55.36 MB

   Sample directories in JPG root: ['p10', 'index.html', 'p14']
   Sample files in first directory: ['p10900906', 'p10005024', 'p10509699']

✅ CXR Data Test Complete!


## CXR File Path Helpers

Helper functions to get file paths for JPG images and their corresponding TXT notes.


In [14]:
def get_cxr_jpg_path(subject_id: int, study_id: int, dicom_id: str) -> Path:
    """
    Get the file path for a CXR JPG image.
    
    Args:
        subject_id: Patient subject ID
        study_id: Study ID
        dicom_id: DICOM ID (used as filename)
        
    Returns:
        Path to the JPG file
    """
    # Format: mimic-cxr-jpg/2.0.0/files/p{first_two_digits}/p{subject_id}/s{study_id}/{dicom_id}.jpg
    subject_str = str(subject_id)
    first_two_digits = subject_str[:2]
    jpg_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "files" / f"p{first_two_digits}" / f"p{subject_id}" / f"s{study_id}" / f"{dicom_id}.jpg"
    return jpg_path


def get_cxr_txt_path(subject_id: int, study_id: int) -> Path:
    """
    Get the file path for a CXR TXT note file.
    
    Args:
        subject_id: Patient subject ID
        study_id: Study ID
        
    Returns:
        Path to the TXT note file
    """
    # Format: mimic-cxr/2.0.0/files/p{first_two_digits}/p{subject_id}/s{study_id}.txt
    subject_str = str(subject_id)
    first_two_digits = subject_str[:2]
    txt_path = MIMIC_CXR_PATH / "2.0.0" / "files" / f"p{first_two_digits}" / f"p{subject_id}" / f"s{study_id}.txt"
    return txt_path


def get_cxr_jpg_txt_pair(subject_id: int, study_id: int, dicom_id: str) -> tuple[Path, Path]:
    """
    Get both JPG image path and corresponding TXT note path for a CXR record.
    
    Args:
        subject_id: Patient subject ID
        study_id: Study ID
        dicom_id: DICOM ID (used as JPG filename)
        
    Returns:
        Tuple of (jpg_path, txt_path)
    """
    jpg_path = get_cxr_jpg_path(subject_id, study_id, dicom_id)
    txt_path = get_cxr_txt_path(subject_id, study_id)
    return jpg_path, txt_path


def verify_cxr_files_exist(subject_id: int, study_id: int, dicom_id: str) -> tuple[bool, bool]:
    """
    Verify if both JPG and TXT files exist for a CXR record.
    
    Args:
        subject_id: Patient subject ID
        study_id: Study ID
        dicom_id: DICOM ID
        
    Returns:
        Tuple of (jpg_exists, txt_exists)
    """
    jpg_path, txt_path = get_cxr_jpg_txt_pair(subject_id, study_id, dicom_id)
    return jpg_path.exists(), txt_path.exists()


In [15]:
# Test the file path helper functions
print("=" * 70)
print("Testing CXR File Path Helpers")
print("=" * 70)

# Use a sample from metadata
if 'cxr_metadata' in globals():
    sample = cxr_metadata.iloc[0]
    subject_id = sample['subject_id']
    study_id = sample['study_id']
    dicom_id = sample['dicom_id']
    
    print(f"\nSample record:")
    print(f"  Subject ID: {subject_id}")
    print(f"  Study ID: {study_id}")
    print(f"  DICOM ID: {dicom_id}")
    
    # Get file paths
    jpg_path = get_cxr_jpg_path(subject_id, study_id, dicom_id)
    txt_path = get_cxr_txt_path(subject_id, study_id)
    
    print(f"\n  JPG path: {jpg_path}")
    print(f"  JPG exists: {jpg_path.exists()}")
    
    print(f"\n  TXT path: {txt_path}")
    print(f"  TXT exists: {txt_path.exists()}")
    
    # Verify both exist
    jpg_exists, txt_exists = verify_cxr_files_exist(subject_id, study_id, dicom_id)
    print(f"\n  Both files exist: {jpg_exists and txt_exists}")
    
    if jpg_exists and txt_exists:
        print(f"\n  ✅ Sample files verified successfully!")
        print(f"  This is a valid JPG-TXT pair for processing.")
    else:
        print(f"\n  ⚠️  Some files are missing. Check paths.")
        
else:
    print("\n⚠️  Please run the CXR metadata loading cell first.")


Testing CXR File Path Helpers

Sample record:
  Subject ID: 10000032
  Study ID: 50414267
  DICOM ID: 02aa804e-bde0afdd-112c0b34-7bc16630-4e384014

  JPG path: /data/xiaochen/physionet.org/files/mimic-cxr-jpg/2.0.0/files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg
  JPG exists: True

  TXT path: /data/xiaochen/physionet.org/files/mimic-cxr/2.0.0/files/p10/p10000032/s50414267.txt
  TXT exists: True

  Both files exist: True

  ✅ Sample files verified successfully!
  This is a valid JPG-TXT pair for processing.


## CXR Metadata Columns and Admission Identification

Understanding the CXR metadata structure and how to identify patient admissions/visits.


In [16]:
# Load and analyze CXR metadata columns
print("=" * 70)
print("CXR Metadata Columns")
print("=" * 70)

# Load metadata if not already loaded
if 'cxr_metadata' not in globals():
    cxr_metadata, _, _, _, _ = load_mimiciv_cxr(lazy=True)

print(f"\nTotal columns: {len(cxr_metadata.columns)}")
print(f"\nColumn names and descriptions:")
print("\n1.  dicom_id - Unique identifier for each CXR image")
print("2.  subject_id - Patient identifier")
print("3.  study_id - Unique study identifier (one study can have multiple images)")
print("4.  PerformedProcedureStepDescription - Procedure description")
print("5.  ViewPosition - Image view (PA, AP, LATERAL, etc.)")
print("6.  Rows - Image height in pixels")
print("7.  Columns - Image width in pixels")
print("8.  StudyDate - Date of the study (YYYYMMDD format)")
print("9.  StudyTime - Time of the study (HHMMSS.SSS format)")
print("10. ProcedureCodeSequence_CodeMeaning - Procedure code meaning")
print("11. ViewCodeSequence_CodeMeaning - View code meaning")
print("12. PatientOrientationCodeSequence_CodeMeaning - Patient orientation")

print(f"\n" + "=" * 70)
print("Sample Data")
print("=" * 70)
print(cxr_metadata.head(3))


CXR Metadata Columns

Total columns: 12

Column names and descriptions:

1.  dicom_id - Unique identifier for each CXR image
2.  subject_id - Patient identifier
3.  study_id - Unique study identifier (one study can have multiple images)
4.  PerformedProcedureStepDescription - Procedure description
5.  ViewPosition - Image view (PA, AP, LATERAL, etc.)
6.  Rows - Image height in pixels
7.  Columns - Image width in pixels
8.  StudyDate - Date of the study (YYYYMMDD format)
9.  StudyTime - Time of the study (HHMMSS.SSS format)
10. ProcedureCodeSequence_CodeMeaning - Procedure code meaning
11. ViewCodeSequence_CodeMeaning - View code meaning
12. PatientOrientationCodeSequence_CodeMeaning - Patient orientation

Sample Data
                                       dicom_id  subject_id  study_id  \
0  02aa804e-bde0afdd-112c0b34-7bc16630-4e384014    10000032  50414267   
1  174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962    10000032  50414267   
2  2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab    100

In [17]:
# Understanding how to identify admissions from CXR metadata
print("=" * 70)
print("Identifying Admissions from CXR Metadata")
print("=" * 70)

print("\nKey Relationships:")
print("  - subject_id: Identifies the patient")
print("  - study_id: Identifies a unique study/visit (one study can have multiple images)")
print("  - dicom_id: Identifies individual images within a study")
print("  - StudyDate + StudyTime: Temporal information for when the study was performed")

print("\n" + "=" * 70)
print("Statistics")
print("=" * 70)
print(f"Total subjects: {cxr_metadata['subject_id'].nunique()}")
print(f"Total studies: {cxr_metadata['study_id'].nunique()}")
print(f"Total images (DICOM IDs): {len(cxr_metadata)}")
print(f"Average images per study: {len(cxr_metadata) / cxr_metadata['study_id'].nunique():.2f}")

# Convert StudyDate to datetime for analysis
cxr_metadata['StudyDate_dt'] = pd.to_datetime(
    cxr_metadata['StudyDate'].astype(str), 
    format='%Y%m%d', 
    errors='coerce'
)

print(f"\nDate range: {cxr_metadata['StudyDate_dt'].min().date()} to {cxr_metadata['StudyDate_dt'].max().date()}")

print("\n" + "=" * 70)
print("How to Identify Admissions")
print("=" * 70)
print("\nOption 1: By study_id")
print("  - Each study_id represents one CXR examination/visit")
print("  - Multiple images (dicom_ids) can belong to the same study_id")
print("  - Example: One study might have PA and LATERAL views (2 dicom_ids, 1 study_id)")

print("\nOption 2: By StudyDate")
print("  - Group studies by subject_id + StudyDate")
print("  - Studies on the same day might be part of the same admission")
print("  - Note: This is approximate - a patient might have multiple admissions on the same day")

print("\nOption 3: By StudyDate + StudyTime (within a time window)")
print("  - Group studies within a certain time window (e.g., within 24 hours)")
print("  - More accurate for identifying same admission")

print("\n" + "=" * 70)
print("Sample Patient Analysis")
print("=" * 70)

# Sample patient
sample_subject = cxr_metadata['subject_id'].iloc[0]
patient_data = cxr_metadata[cxr_metadata['subject_id'] == sample_subject].copy()
patient_data = patient_data.sort_values(['StudyDate', 'StudyTime'])

print(f"\nPatient {sample_subject}:")
print(f"  Total studies: {patient_data['study_id'].nunique()}")
print(f"  Total images: {len(patient_data)}")
print(f"  Date range: {patient_data['StudyDate_dt'].min().date()} to {patient_data['StudyDate_dt'].max().date()}")

print(f"\n  Studies grouped by date (potential admissions):")
for date, group in patient_data.groupby('StudyDate'):
    date_str = pd.to_datetime(str(date), format='%Y%m%d').date()
    study_ids = group['study_id'].unique()
    dicom_count = len(group)
    print(f"    Date: {date_str} - {len(study_ids)} study(ies), {dicom_count} image(s)")
    print(f"      Study IDs: {list(study_ids)}")


Identifying Admissions from CXR Metadata

Key Relationships:
  - subject_id: Identifies the patient
  - study_id: Identifies a unique study/visit (one study can have multiple images)
  - dicom_id: Identifies individual images within a study
  - StudyDate + StudyTime: Temporal information for when the study was performed

Statistics
Total subjects: 65379
Total studies: 227835
Total images (DICOM IDs): 377110
Average images per study: 1.66

Date range: 2110-01-11 to 2208-12-08

How to Identify Admissions

Option 1: By study_id
  - Each study_id represents one CXR examination/visit
  - Multiple images (dicom_ids) can belong to the same study_id
  - Example: One study might have PA and LATERAL views (2 dicom_ids, 1 study_id)

Option 2: By StudyDate
  - Group studies by subject_id + StudyDate
  - Studies on the same day might be part of the same admission
  - Note: This is approximate - a patient might have multiple admissions on the same day

Option 3: By StudyDate + StudyTime (within a ti

## EHR Data Structure and Admission Identification

Understanding the EHR data structure and how admissions are identified.


In [18]:
# Load and analyze EHR admissions
print("=" * 70)
print("MIMIC-IV EHR Admissions Columns")
print("=" * 70)

# Load admissions if not already loaded
if 'ehr_admissions' not in globals():
    ehr_admissions, ehr_patients, ehr_diagnoses, _, _, _ = load_mimiciv_ehr(lazy=True)
else:
    # Use existing if already loaded
    pass

print(f"\nTotal columns: {len(ehr_admissions.columns)}")
print(f"\nColumn names and descriptions:")
print("\n1.  subject_id - Patient identifier")
print("2.  hadm_id - Hospital admission identifier (unique per admission)")
print("3.  admittime - Admission date and time")
print("4.  dischtime - Discharge date and time")
print("5.  deathtime - Death time (if applicable)")
print("6.  admission_type - Type of admission (e.g., URGENT, EW EMER.)")
print("7.  admit_provider_id - Provider ID at admission")
print("8.  admission_location - Location where patient was admitted from")
print("9.  discharge_location - Location where patient was discharged to")
print("10. insurance - Insurance type")
print("11. language - Patient language")
print("12. marital_status - Marital status")
print("13. race - Patient race")
print("14. edregtime - Emergency department registration time")
print("15. edouttime - Emergency department discharge time")
print("16. hospital_expire_flag - Flag indicating if patient expired in hospital")

print(f"\n" + "=" * 70)
print("Sample Admission Records")
print("=" * 70)
print(ehr_admissions.head(3))


MIMIC-IV EHR Admissions Columns

Total columns: 16

Column names and descriptions:

1.  subject_id - Patient identifier
2.  hadm_id - Hospital admission identifier (unique per admission)
3.  admittime - Admission date and time
4.  dischtime - Discharge date and time
5.  deathtime - Death time (if applicable)
6.  admission_type - Type of admission (e.g., URGENT, EW EMER.)
7.  admit_provider_id - Provider ID at admission
8.  admission_location - Location where patient was admitted from
9.  discharge_location - Location where patient was discharged to
10. insurance - Insurance type
11. language - Patient language
12. marital_status - Marital status
13. race - Patient race
14. edregtime - Emergency department registration time
15. edouttime - Emergency department discharge time
16. hospital_expire_flag - Flag indicating if patient expired in hospital

Sample Admission Records
   subject_id   hadm_id            admittime            dischtime deathtime  \
0    10000032  22595853  2180-05-06 

In [19]:
# Understanding EHR admissions structure
print("=" * 70)
print("Understanding EHR Admissions")
print("=" * 70)

# Convert datetime columns
ehr_admissions['admittime'] = pd.to_datetime(ehr_admissions['admittime'])
ehr_admissions['dischtime'] = pd.to_datetime(ehr_admissions['dischtime'])

print("\nKey Identifiers:")
print("  - subject_id: Patient identifier (same across all modalities)")
print("  - hadm_id: Hospital admission ID (unique per admission, used to link EHR data)")

print("\n" + "=" * 70)
print("Statistics")
print("=" * 70)
print(f"Total admissions: {len(ehr_admissions)}")
print(f"Unique subjects: {ehr_admissions['subject_id'].nunique()}")
print(f"Unique hadm_ids: {ehr_admissions['hadm_id'].nunique()}")
print(f"Average admissions per patient: {len(ehr_admissions) / ehr_admissions['subject_id'].nunique():.2f}")

print(f"\nAdmission date range:")
print(f"  From: {ehr_admissions['admittime'].min()}")
print(f"  To: {ehr_admissions['dischtime'].max()}")

print("\n" + "=" * 70)
print("How to Identify Admissions")
print("=" * 70)
print("\nEHR admissions are clearly identified by:")
print("  - hadm_id: Each hadm_id represents one hospital admission")
print("  - subject_id + hadm_id: Uniquely identifies a patient's admission")
print("  - admittime to dischtime: Time window for the admission")
print("\nNote: hadm_id is the primary key for linking:")
print("  - Diagnoses (diagnoses_icd.csv)")
print("  - Procedures (procedures_icd.csv)")
print("  - Lab events (labevents.csv)")
print("  - Prescriptions (prescriptions.csv)")
print("  - And other EHR data to admissions")

print("\n" + "=" * 70)
print("Sample Patient Analysis")
print("=" * 70)

# Sample patient
sample_subject = ehr_admissions['subject_id'].iloc[0]
patient_adms = ehr_admissions[ehr_admissions['subject_id'] == sample_subject].sort_values('admittime')

print(f"\nPatient {sample_subject} has {len(patient_adms)} admissions:")
for idx, adm in patient_adms.iterrows():
    print(f"\n  HADM {adm['hadm_id']}:")
    print(f"    Admit: {adm['admittime']}")
    print(f"    Discharge: {adm['dischtime']}")
    print(f"    Duration: {adm['dischtime'] - adm['admittime']}")
    if 'admission_type' in adm:
        print(f"    Type: {adm['admission_type']}")
    if 'admission_location' in adm:
        print(f"    Location: {adm['admission_location']}")


Understanding EHR Admissions

Key Identifiers:
  - subject_id: Patient identifier (same across all modalities)
  - hadm_id: Hospital admission ID (unique per admission, used to link EHR data)

Statistics
Total admissions: 431231
Unique subjects: 180733
Unique hadm_ids: 431231
Average admissions per patient: 2.39

Admission date range:
  From: 2105-10-04 17:26:00
  To: 2212-04-12 14:06:00

How to Identify Admissions

EHR admissions are clearly identified by:
  - hadm_id: Each hadm_id represents one hospital admission
  - subject_id + hadm_id: Uniquely identifies a patient's admission
  - admittime to dischtime: Time window for the admission

Note: hadm_id is the primary key for linking:
  - Diagnoses (diagnoses_icd.csv)
  - Procedures (procedures_icd.csv)
  - Lab events (labevents.csv)
  - Prescriptions (prescriptions.csv)
  - And other EHR data to admissions

Sample Patient Analysis

Patient 10000032 has 4 admissions:

  HADM 22595853:
    Admit: 2180-05-06 22:23:00
    Discharge: 2180

In [20]:
# Check available EHR tables
import os
hosp_path = MIMICIV_PATH / "2.2" / "hosp"
tables = [f for f in os.listdir(hosp_path) if f.endswith('.csv')]

print("=" * 70)
print("Available EHR Tables")
print("=" * 70)
print(f"\nTotal tables in hosp/: {len(tables)}")
print(f"\nKey tables for multimodal data:")
print("\n1. Core Tables:")
print("   - admissions.csv - Hospital admissions")
print("   - patients.csv - Patient demographics")
print("   - diagnoses_icd.csv - ICD diagnoses (linked via hadm_id)")
print("   - procedures_icd.csv - ICD procedures (linked via hadm_id)")

print("\n2. Clinical Data Tables:")
print("   - labevents.csv - Laboratory results (linked via hadm_id)")
print("   - prescriptions.csv - Medications (linked via hadm_id)")
print("   - pharmacy.csv - Pharmacy data")

print("\n3. Other Tables:")
for table in sorted(tables)[:10]:
    print(f"   - {table}")
print(f"   ... and {len(tables) - 10} more")

print("\n" + "=" * 70)
print("Key Relationships")
print("=" * 70)
print("\nAll EHR data is linked via:")
print("  - subject_id: Patient identifier (used across all modalities)")
print("  - hadm_id: Hospital admission ID (used to link EHR data within an admission)")
print("\nNote: CXR uses study_id (unique to CXR), not hadm_id")
print("      To match CXR to admissions, use subject_id + StudyDate/StudyTime")
print("      and match to admission time window (admittime to dischtime)")


Available EHR Tables

Total tables in hosp/: 22

Key tables for multimodal data:

1. Core Tables:
   - admissions.csv - Hospital admissions
   - patients.csv - Patient demographics
   - diagnoses_icd.csv - ICD diagnoses (linked via hadm_id)
   - procedures_icd.csv - ICD procedures (linked via hadm_id)

2. Clinical Data Tables:
   - labevents.csv - Laboratory results (linked via hadm_id)
   - prescriptions.csv - Medications (linked via hadm_id)
   - pharmacy.csv - Pharmacy data

3. Other Tables:
   - admissions.csv
   - d_hcpcs.csv
   - d_icd_diagnoses.csv
   - d_icd_procedures.csv
   - d_labitems.csv
   - diagnoses_icd.csv
   - drgcodes.csv
   - emar.csv
   - emar_detail.csv
   - hcpcsevents.csv
   ... and 12 more

Key Relationships

All EHR data is linked via:
  - subject_id: Patient identifier (used across all modalities)
  - hadm_id: Hospital admission ID (used to link EHR data within an admission)

Note: CXR uses study_id (unique to CXR), not hadm_id
      To match CXR to admission

## ECG Data Structure and Visit Identification

Understanding the MIMIC-IV ECG dataset structure and how to identify visits/admissions.


In [21]:
# Load and analyze ECG data
print("=" * 70)
print("MIMIC-IV ECG Dataset Structure")
print("=" * 70)

# Load ECG data if not already loaded
if 'ecg_report' not in globals():
    _, _, ecg_report, ecg_record_list = load_mimiciv_ecg(lazy=False)
else:
    # Load record_list if not loaded
    if 'ecg_record_list' not in globals():
        ecg_base = MIMIC_IV_ECG_PATH / "1.0"
        ecg_record_list = pd.read_csv(ecg_base / "record_list.csv")

print(f"\nTotal columns in machine_measurements: {len(ecg_report.columns)}")
print(f"\nColumn names and descriptions:")
print("\n1.  subject_id - Patient identifier (shared with other modalities)")
print("2.  study_id - Study identifier (unique per ECG record)")
print("3.  cart_id - Cart/device identifier for ECG recording")
print("4.  ecg_time - Time of ECG recording (key temporal identifier)")
print("\n5-22. report_0 to report_17 - ECG interpretation text (segmented)")
print("\n23-24. bandwidth, filtering - ECG signal parameters")
print("25.  rr_interval - RR interval measurement")
print("26-30. p_onset, p_end, qrs_onset, qrs_end, t_end - Waveform measurements")
print("31-33. p_axis, qrs_axis, t_axis - Electrical axis measurements")

print(f"\n" + "=" * 70)
print("Sample ECG Records")
print("=" * 70)
print(ecg_report.head(3))


MIMIC-IV ECG Dataset Structure

Total columns in machine_measurements: 33

Column names and descriptions:

1.  subject_id - Patient identifier (shared with other modalities)
2.  study_id - Study identifier (unique per ECG record)
3.  cart_id - Cart/device identifier for ECG recording
4.  ecg_time - Time of ECG recording (key temporal identifier)

5-22. report_0 to report_17 - ECG interpretation text (segmented)

23-24. bandwidth, filtering - ECG signal parameters
25.  rr_interval - RR interval measurement
26-30. p_onset, p_end, qrs_onset, qrs_end, t_end - Waveform measurements
31-33. p_axis, qrs_axis, t_axis - Electrical axis measurements

Sample ECG Records
   subject_id  study_id  cart_id             ecg_time           report_0  \
0    10000032  40689238  6848296  2180-07-23 08:44:00       Sinus rhythm   
1    10000032  44458630  6848296  2180-07-23 09:54:00       Sinus rhythm   
2    10000032  49036311  6376932  2180-08-06 09:07:00  Sinus tachycardia   

                            

In [22]:
# Understanding ECG visit identification
print("=" * 70)
print("Understanding ECG Visit/Admission Identification")
print("=" * 70)

# Convert ecg_time to datetime
ecg_report['ecg_time'] = pd.to_datetime(ecg_report['ecg_time'])

print("\nKey Identifiers in ECG:")
print("  - subject_id: Patient identifier (shared across all modalities)")
print("  - study_id: Study identifier (unique per ECG record - 1:1 mapping)")
print("  - cart_id: Cart/device identifier")
print("  - ecg_time: Temporal identifier (key for matching to admissions)")

print("\n" + "=" * 70)
print("Statistics")
print("=" * 70)
print(f"Total ECG records: {len(ecg_report)}")
print(f"Unique subjects: {ecg_report['subject_id'].nunique()}")
print(f"Unique studies: {ecg_report['study_id'].nunique()}")
print(f"Unique cart IDs: {ecg_report['cart_id'].nunique()}")
print(f"\nNote: study_id in ECG is unique per record (1:1 mapping)")
print(f"      This is different from CXR where one study_id can have multiple images")

print(f"\nECG time range:")
print(f"  From: {ecg_report['ecg_time'].min()}")
print(f"  To: {ecg_report['ecg_time'].max()}")

print("\n" + "=" * 70)
print("How to Identify Visits/Admissions in ECG")
print("=" * 70)
print("\nOption 1: By study_id")
print("  - Each study_id = one ECG recording")
print("  - study_id is unique per record (different from CXR)")

print("\nOption 2: By ecg_time (temporal matching)")
print("  - Use subject_id + ecg_time to match to admissions")
print("  - Match ECG records to EHR admissions if:")
print("    admittime <= ecg_time <= dischtime")

print("\nOption 3: By ecg_time + time window")
print("  - Group ECG records within a time window (e.g., 24 hours)")
print("  - More flexible for identifying same visit")

print("\n" + "=" * 70)
print("Sample Patient Analysis")
print("=" * 70)

# Sample patient
sample_subject = ecg_report['subject_id'].iloc[0]
patient_ecg = ecg_report[ecg_report['subject_id'] == sample_subject].copy()
patient_ecg = patient_ecg.sort_values('ecg_time')

print(f"\nPatient {sample_subject} has {len(patient_ecg)} ECG records")
print(f"  Unique studies: {patient_ecg['study_id'].nunique()}")
print(f"  ECG time range: {patient_ecg['ecg_time'].min()} to {patient_ecg['ecg_time'].max()}")

print(f"\n  ECG records grouped by date (potential visits):")
patient_ecg['ecg_date'] = patient_ecg['ecg_time'].dt.date
for date, group in patient_ecg.groupby('ecg_date'):
    print(f"    Date: {date} - {len(group)} record(s)")
    print(f"      Study IDs: {list(group['study_id'].unique())}")
    print(f"      Times: {group['ecg_time'].dt.strftime('%H:%M:%S').tolist()}")


Understanding ECG Visit/Admission Identification

Key Identifiers in ECG:
  - subject_id: Patient identifier (shared across all modalities)
  - study_id: Study identifier (unique per ECG record - 1:1 mapping)
  - cart_id: Cart/device identifier
  - ecg_time: Temporal identifier (key for matching to admissions)

Statistics
Total ECG records: 800035
Unique subjects: 161352
Unique studies: 800035
Unique cart IDs: 156

Note: study_id in ECG is unique per record (1:1 mapping)
      This is different from CXR where one study_id can have multiple images

ECG time range:
  From: 2087-05-08 00:05:00
  To: 2224-04-14 02:31:00

How to Identify Visits/Admissions in ECG

Option 1: By study_id
  - Each study_id = one ECG recording
  - study_id is unique per record (different from CXR)

Option 2: By ecg_time (temporal matching)
  - Use subject_id + ecg_time to match to admissions
  - Match ECG records to EHR admissions if:
    admittime <= ecg_time <= dischtime

Option 3: By ecg_time + time window
  

In [23]:
# Check ECG report structure
print("=" * 70)
print("ECG Report Structure")
print("=" * 70)

# Combine report fields into full text
def combine_ecg_report(row):
    """Combine report_0 through report_17 into full text"""
    report_parts = []
    for i in range(18):
        val = row[f'report_{i}']
        if pd.notna(val) and str(val).strip():
            report_parts.append(str(val).strip())
    return ', '.join(report_parts) if report_parts else "No ECG report available"

# Sample report
sample = ecg_report.iloc[0]
full_report = combine_ecg_report(sample)

print(f"\nSample ECG record:")
print(f"  Subject ID: {sample['subject_id']}")
print(f"  Study ID: {sample['study_id']}")
print(f"  Cart ID: {sample['cart_id']}")
print(f"  ECG Time: {sample['ecg_time']}")
print(f"\n  Full Report:")
print(f"  {full_report}")

print(f"\n" + "=" * 70)
print("ECG Record List")
print("=" * 70)
print(f"\nrecord_list.csv provides file paths for waveform data")
print(f"Columns: {list(ecg_record_list.columns)}")
print(f"Total records: {len(ecg_record_list)}")
print(f"\nSample:")
print(ecg_record_list.head(3))


ECG Report Structure

Sample ECG record:
  Subject ID: 10000032
  Study ID: 40689238
  Cart ID: 6848296
  ECG Time: 2180-07-23 08:44:00

  Full Report:
  Sinus rhythm, Possible right atrial abnormality, Borderline ECG

ECG Record List

record_list.csv provides file paths for waveform data
Columns: ['subject_id', 'study_id', 'file_name', 'ecg_time', 'path']
Total records: 800035

Sample:
   subject_id  study_id  file_name             ecg_time  \
0    10000032  40689238   40689238  2180-07-23 08:44:00   
1    10000032  44458630   44458630  2180-07-23 09:54:00   
2    10000032  49036311   49036311  2180-08-06 09:07:00   

                                       path  
0  files/p1000/p10000032/s40689238/40689238  
1  files/p1000/p10000032/s44458630/44458630  
2  files/p1000/p10000032/s49036311/49036311  


d. 

In [24]:
# Display sample ECG reports
print("=" * 70)
print("Sample ECG Reports")
print("=" * 70)

# Function to combine report fields into full text
def combine_ecg_report(row):
    """Combine report_0 through report_17 into full text"""
    report_parts = []
    for i in range(18):
        val = row[f'report_{i}']
        if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
            report_parts.append(str(val).strip())
    return ', '.join(report_parts) if report_parts else "No ECG report available"

# Show multiple samples
print("\nShowing 5 sample ECG reports:\n")
for i in range(5):
    sample = ecg_report.iloc[i]
    full_report = combine_ecg_report(sample)
    
    print("-" * 70)
    print(f"Sample {i+1}:")
    print(f"  Subject ID: {sample['subject_id']}")
    print(f"  Study ID: {sample['study_id']}")
    print(f"  Cart ID: {sample['cart_id']}")
    print(f"  ECG Time: {sample['ecg_time']}")
    print(f"\n  Full Report:")
    print(f"  {full_report}")
    
    # Show individual report fields that are not empty
    non_empty_fields = []
    for j in range(18):
        val = sample[f'report_{j}']
        if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
            non_empty_fields.append(f"report_{j}: {val}")
    
    if non_empty_fields:
        print(f"\n  Individual Report Fields:")
        for field in non_empty_fields:
            print(f"    {field}")
    
    # Show measurements if available
    measurements = []
    if pd.notna(sample.get('rr_interval')):
        measurements.append(f"RR Interval: {sample['rr_interval']} ms")
    if pd.notna(sample.get('p_axis')):
        measurements.append(f"P Axis: {sample['p_axis']}°")
    if pd.notna(sample.get('qrs_axis')):
        measurements.append(f"QRS Axis: {sample['qrs_axis']}°")
    if pd.notna(sample.get('t_axis')):
        measurements.append(f"T Axis: {sample['t_axis']}°")
    
    if measurements:
        print(f"\n  Measurements:")
        for meas in measurements:
            print(f"    {meas}")
    print()


Sample ECG Reports

Showing 5 sample ECG reports:

----------------------------------------------------------------------
Sample 1:
  Subject ID: 10000032
  Study ID: 40689238
  Cart ID: 6848296
  ECG Time: 2180-07-23 08:44:00

  Full Report:
  Sinus rhythm, Possible right atrial abnormality, Borderline ECG

  Individual Report Fields:
    report_0: Sinus rhythm
    report_1: Possible right atrial abnormality
    report_3: Borderline ECG

  Measurements:
    RR Interval: 659 ms
    P Axis: 81°
    QRS Axis: 77°
    T Axis: 79°

----------------------------------------------------------------------
Sample 2:
  Subject ID: 10000032
  Study ID: 44458630
  Cart ID: 6848296
  ECG Time: 2180-07-23 09:54:00

  Full Report:
  Sinus rhythm, Possible right atrial abnormality, Borderline ECG

  Individual Report Fields:
    report_0: Sinus rhythm
    report_1: Possible right atrial abnormality
    report_3: Borderline ECG

  Measurements:
    RR Interval: 722 ms
    P Axis: 77°
    QRS Axis: 75°


## Cross-Modality Comparison and Patient Matching

Comparing metadata structures across EHR, CXR, and ECG, and demonstrating how to match patient records.


In [25]:
# Compare metadata structures across modalities
print("=" * 70)
print("Cross-Modality Metadata Comparison")
print("=" * 70)

# Ensure datasets are loaded
if 'ehr_admissions' not in globals():
    ehr_admissions, _, _, _, _, _ = load_mimiciv_ehr(lazy=True)
    ehr_admissions['admittime'] = pd.to_datetime(ehr_admissions['admittime'])
    ehr_admissions['dischtime'] = pd.to_datetime(ehr_admissions['dischtime'])

if 'cxr_metadata' not in globals():
    cxr_metadata, _, _, _, _ = load_mimiciv_cxr(lazy=True)
    cxr_metadata['StudyDate_dt'] = pd.to_datetime(cxr_metadata['StudyDate'].astype(str), format='%Y%m%d', errors='coerce')
    cxr_metadata['StudyTime_parsed'] = pd.to_timedelta(
        cxr_metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[:2] + ':' + 
        cxr_metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[2:4] + ':' + 
        cxr_metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[4:6], 
        errors='coerce'
    )
    cxr_metadata['study_datetime'] = cxr_metadata['StudyDate_dt'] + cxr_metadata['StudyTime_parsed']

if 'ecg_report' not in globals():
    _, _, ecg_report, _ = load_mimiciv_ecg(lazy=False)
    ecg_report['ecg_time'] = pd.to_datetime(ecg_report['ecg_time'])

# Find common subjects
ehr_subjects = set(ehr_admissions['subject_id'].unique())
cxr_subjects = set(cxr_metadata['subject_id'].unique())
ecg_subjects = set(ecg_report['subject_id'].unique())
common_subjects = ehr_subjects & cxr_subjects & ecg_subjects

print(f"\nDataset Statistics:")
print(f"  EHR subjects: {len(ehr_subjects)}")
print(f"  CXR subjects: {len(cxr_subjects)}")
print(f"  ECG subjects: {len(ecg_subjects)}")
print(f"  Common subjects (all three): {len(common_subjects)} ({len(common_subjects)/max(len(ehr_subjects), len(cxr_subjects), len(ecg_subjects))*100:.1f}%)")

print(f"\n" + "=" * 70)
print("Key Identifiers Comparison")
print("=" * 70)

print(f"\nEHR:")
print(f"  - subject_id: Patient identifier (shared)")
print(f"  - hadm_id: Hospital admission ID (unique per admission)")
print(f"  - admittime, dischtime: Admission time window")

print(f"\nCXR:")
print(f"  - subject_id: Patient identifier (shared)")
print(f"  - study_id: Study ID (unique to CXR, one study can have multiple images)")
print(f"  - dicom_id: Individual image ID")
print(f"  - StudyDate, StudyTime: Temporal information")

print(f"\nECG:")
print(f"  - subject_id: Patient identifier (shared)")
print(f"  - study_id: Study ID (unique to ECG, 1:1 with records)")
print(f"  - cart_id: Cart/device ID")
print(f"  - ecg_time: Temporal information")

print(f"\n" + "=" * 70)
print("Matching Strategy")
print("=" * 70)
print(f"\n1. Common identifier: subject_id (shared across all modalities)")
print(f"\n2. Temporal matching:")
print(f"   - EHR: admittime <= event_time <= dischtime")
print(f"   - CXR: StudyDate/StudyTime matched to admission window")
print(f"   - ECG: ecg_time matched to admission window")
print(f"\n3. Important notes:")
print(f"   - study_id is different across modalities:")
print(f"     * CXR study_id: unique to CXR, one study can have multiple images")
print(f"     * ECG study_id: unique to ECG, one study = one record")
print(f"   - No direct hadm_id in CXR or ECG (need temporal matching)")
print(f"   - Time is the major factor for defining visits when merging")


Cross-Modality Metadata Comparison

Dataset Statistics:
  EHR subjects: 180733
  CXR subjects: 65379
  ECG subjects: 161352
  Common subjects (all three): 47187 (26.1%)

Key Identifiers Comparison

EHR:
  - subject_id: Patient identifier (shared)
  - hadm_id: Hospital admission ID (unique per admission)
  - admittime, dischtime: Admission time window

CXR:
  - subject_id: Patient identifier (shared)
  - study_id: Study ID (unique to CXR, one study can have multiple images)
  - dicom_id: Individual image ID
  - StudyDate, StudyTime: Temporal information

ECG:
  - subject_id: Patient identifier (shared)
  - study_id: Study ID (unique to ECG, 1:1 with records)
  - cart_id: Cart/device ID
  - ecg_time: Temporal information

Matching Strategy

1. Common identifier: subject_id (shared across all modalities)

2. Temporal matching:
   - EHR: admittime <= event_time <= dischtime
   - CXR: StudyDate/StudyTime matched to admission window
   - ECG: ecg_time matched to admission window

3. Importan

In [26]:
# Find a patient with all three modalities in the same admission
print("=" * 70)
print("Example: Patient with All Three Modalities")
print("=" * 70)

# Function to match CXR and ECG to an admission
def match_modalities_to_admission(subject_id, hadm_id, admittime, dischtime):
    """Match CXR and ECG records to a specific admission"""
    # Get patient's CXR and ECG
    patient_cxr = cxr_metadata[cxr_metadata['subject_id'] == subject_id].copy()
    patient_ecg = ecg_report[ecg_report['subject_id'] == subject_id].copy()
    
    # Match CXR within admission window
    matching_cxr = patient_cxr[
        (patient_cxr['study_datetime'] >= admittime) & 
        (patient_cxr['study_datetime'] <= dischtime)
    ]
    
    # Match ECG within admission window
    matching_ecg = patient_ecg[
        (patient_ecg['ecg_time'] >= admittime) & 
        (patient_ecg['ecg_time'] <= dischtime)
    ]
    
    return matching_cxr, matching_ecg

# Find a patient with all three in same admission
found_patient = None
for subject_id in list(common_subjects)[:200]:  # Check first 200
    patient_ehr = ehr_admissions[ehr_admissions['subject_id'] == subject_id].sort_values('admittime')
    
    for _, adm in patient_ehr.iterrows():
        matching_cxr, matching_ecg = match_modalities_to_admission(
            subject_id, adm['hadm_id'], adm['admittime'], adm['dischtime']
        )
        
        if len(matching_cxr) > 0 and len(matching_ecg) > 0:
            found_patient = {
                'subject_id': subject_id,
                'hadm_id': adm['hadm_id'],
                'admission': adm,
                'cxr': matching_cxr,
                'ecg': matching_ecg
            }
            break
    
    if found_patient:
        break

if found_patient:
    print(f"\n✅ Found patient with all three modalities in the same admission!")
    print(f"\nPatient: {found_patient['subject_id']}")
    print(f"HADM ID: {found_patient['hadm_id']}")
    
    adm = found_patient['admission']
    print(f"\nAdmission:")
    print(f"  Admit: {adm['admittime']}")
    print(f"  Discharge: {adm['dischtime']}")
    print(f"  Duration: {adm['dischtime'] - adm['admittime']}")
    if 'admission_type' in adm:
        print(f"  Type: {adm['admission_type']}")
    
    print(f"\nMatching CXR Studies ({len(found_patient['cxr'])} images from {found_patient['cxr']['study_id'].nunique()} studies):")
    for study_id in found_patient['cxr']['study_id'].unique():
        study_cxr = found_patient['cxr'][found_patient['cxr']['study_id'] == study_id]
        print(f"\n  Study ID: {study_id}")
        for idx, cxr in study_cxr.iterrows():
            print(f"    DICOM ID: {cxr['dicom_id'][:40]}...")
            print(f"      Time: {cxr['study_datetime']}")
            print(f"      View: {cxr.get('ViewPosition', 'N/A')}")
    
    print(f"\nMatching ECG Records ({len(found_patient['ecg'])} records):")
    for idx, ecg in found_patient['ecg'].iterrows():
        print(f"  Study ID: {ecg['study_id']}, Cart ID: {ecg['cart_id']}")
        print(f"    Time: {ecg['ecg_time']}")
        # Combine report
        report_parts = []
        for i in range(18):
            val = ecg[f'report_{i}']
            if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                report_parts.append(str(val).strip())
        if report_parts:
            print(f"    Report: {', '.join(report_parts)}")
else:
    print("\n⚠️  No exact match found in first 200 patients.")
    print("Showing example with nearby matches...")
    subject_id = list(common_subjects)[0]
    patient_ehr = ehr_admissions[ehr_admissions['subject_id'] == subject_id].sort_values('admittime')
    adm = patient_ehr.iloc[0]
    print(f"\nPatient: {subject_id}, HADM: {adm['hadm_id']}")
    print(f"Admission: {adm['admittime']} to {adm['dischtime']}")
    
    # Show nearby (within 7 days)
    patient_cxr = cxr_metadata[cxr_metadata['subject_id'] == subject_id]
    patient_ecg = ecg_report[ecg_report['subject_id'] == subject_id]
    
    nearby_cxr = patient_cxr[
        (patient_cxr['study_datetime'] >= adm['admittime'] - pd.Timedelta(days=7)) & 
        (patient_cxr['study_datetime'] <= adm['dischtime'] + pd.Timedelta(days=7))
    ]
    nearby_ecg = patient_ecg[
        (patient_ecg['ecg_time'] >= adm['admittime'] - pd.Timedelta(days=7)) & 
        (patient_ecg['ecg_time'] <= adm['dischtime'] + pd.Timedelta(days=7))
    ]
    
    print(f"\nNearby CXR (within 7 days): {len(nearby_cxr)}")
    print(f"Nearby ECG (within 7 days): {len(nearby_ecg)}")


Example: Patient with All Three Modalities


KeyError: 'study_datetime'

## Summary: Patient Record Matching Strategy

Key principles for matching patient records across modalities.


In [ ]:
# Summary of matching strategy
print("=" * 70)
print("Patient Record Matching Strategy - Summary")
print("=" * 70)

print("\n" + "=" * 70)
print("1. EHR Data - Use Admission (hadm_id) as Primary Identifier")
print("=" * 70)
print("""
For EHR data, we use HADM ID (hadm_id) as the primary identifier because:
  - Each admission (hadm_id) represents one hospital visit/stay
  - All EHR data within an admission is linked via hadm_id:
    * ICD diagnosis codes (diagnoses_icd.csv)
    * ICD procedure codes (procedures_icd.csv)
    * Lab results (labevents.csv)
    * Prescriptions (prescriptions.csv)
    * And other clinical data
  - Each admission has a clear time window: admittime to dischtime
  - This provides a well-defined temporal and clinical context
""")

print("\n" + "=" * 70)
print("2. CXR and ECG - Different Study IDs, Use Time for Matching")
print("=" * 70)
print("""
For CXR and ECG data:
  - study_id is unique to each modality (CXR study_id ≠ ECG study_id)
  - CXR study_id: One study can have multiple images (different dicom_ids)
  - ECG study_id: One study = one record (1:1 mapping)
  - study_ids do NOT correspond across modalities
  - Therefore, we CANNOT use study_id to match CXR and ECG directly
""")

print("\n" + "=" * 70)
print("3. Temporal Matching Strategy")
print("=" * 70)
print("""
To match CXR and ECG to EHR admissions, we use TEMPORAL matching:

  Matching Rule:
  ┌─────────────────────────────────────────────────────────┐
  │  CXR or ECG record belongs to admission if:            │
  │                                                          │
  │  admittime <= record_time <= dischtime                   │
  │                                                          │
  │  where:                                                  │
  │  - CXR: record_time = StudyDate + StudyTime             │
  │  - ECG: record_time = ecg_time                          │
  └─────────────────────────────────────────────────────────┘

  Steps:
  1. Start with EHR admission (hadm_id) with known time window
  2. For the same patient (subject_id), find:
     - CXR studies where study_datetime falls within admission window
     - ECG records where ecg_time falls within admission window
  3. Group all matched records under that admission

  This creates a unified patient record structure:
  Patient (subject_id)
    └── Admission (hadm_id)
        ├── EHR data (diagnoses, procedures, labs, prescriptions)
        ├── CXR studies (matched by time)
        └── ECG records (matched by time)
""")

print("\n" + "=" * 70)
print("4. Key Points")
print("=" * 70)
print("""
✓ Common identifier: subject_id (links all modalities)
✓ EHR: hadm_id is the primary grouping unit
✓ CXR/ECG: study_id is modality-specific, not cross-modality
✓ Time is the bridge: temporal matching connects modalities
✓ Each admission can have multiple CXR studies and ECG records
✓ One CXR study can have multiple images (different views)
✓ One ECG study = one record
""")

print("\n" + "=" * 70)
print("Example Structure")
print("=" * 70)
print("""
Patient 17039362
  └── Admission 22414027 (2127-05-07 14:13 to 2127-05-08 19:18)
      ├── EHR Data:
      │   ├── Diagnoses (via hadm_id)
      │   ├── Procedures (via hadm_id)
      │   ├── Lab results (via hadm_id)
      │   └── Prescriptions (via hadm_id)
      │
      ├── CXR Studies (matched by time):
      │   └── Study 55283114 (2027-05-07 20:42)
      │       ├── Image 1: PA view (dicom_id: ...)
      │       └── Image 2: PA view (dicom_id: ...)
      │
      └── ECG Records (matched by time):
          └── Study 40212991 (2027-05-07 18:46)
              └── ECG report: "Sinus rhythm, ..."
""")


## Patient Data in Sequential Format

Displaying a patient's complete data organized chronologically by admission (hadm_id).


In [ ]:
# Display patient data in sequential format
print("=" * 80)
print("PATIENT DATA IN TIME/HADM_ID SEQUENTIAL FORMAT")
print("=" * 80)

# Load diagnoses if needed
if 'ehr_diagnoses' not in globals():
    ehr_diagnoses = pd.read_csv(MIMICIV_PATH / "2.2" / "hosp" / "diagnoses_icd.csv")
    ehr_diagnoses.columns = ehr_diagnoses.columns.str.lower()

# Find patient with all three modalities
def find_patient_with_all_modalities():
    """Find a patient with all three modalities"""
    ehr_subjects = set(ehr_admissions['subject_id'].unique())
    cxr_subjects = set(cxr_metadata['subject_id'].unique())
    ecg_subjects = set(ecg_report['subject_id'].unique())
    common_subjects = list(ehr_subjects & cxr_subjects & ecg_subjects)
    
    for subject_id in common_subjects[:500]:
        patient_ehr = ehr_admissions[ehr_admissions['subject_id'] == subject_id].sort_values('admittime')
        patient_cxr = cxr_metadata[cxr_metadata['subject_id'] == subject_id].sort_values('study_datetime')
        patient_ecg = ecg_report[ecg_report['subject_id'] == subject_id].sort_values('ecg_time')
        
        matches = []
        for _, adm in patient_ehr.iterrows():
            matching_cxr = patient_cxr[
                (patient_cxr['study_datetime'] >= adm['admittime']) & 
                (patient_cxr['study_datetime'] <= adm['dischtime'])
            ]
            matching_ecg = patient_ecg[
                (patient_ecg['ecg_time'] >= adm['admittime']) & 
                (patient_ecg['ecg_time'] <= adm['dischtime'])
            ]
            
            if len(matching_cxr) > 0 or len(matching_ecg) > 0:
                matches.append({
                    'admission': adm,
                    'cxr': matching_cxr,
                    'ecg': matching_ecg
                })
        
        if len(matches) >= 1:
            return {
                'subject_id': subject_id,
                'matches': matches
            }
    return None

# Find patient
patient_data = find_patient_with_all_modalities()

if patient_data:
    subject_id = patient_data['subject_id']
    print(f"\nPatient ID: {subject_id}")
    print(f"\nTotal Admissions with Matched Data: {len(patient_data['matches'])}")
    print("=" * 80)
    
    # Display each admission sequentially
    for idx, match in enumerate(patient_data['matches'], 1):
        adm = match['admission']
        matching_cxr = match['cxr']
        matching_ecg = match['ecg']
        
        print(f"\n{'=' * 80}")
        print(f"ADMISSION #{idx}: HADM ID {adm['hadm_id']}")
        print(f"{'=' * 80}")
        print(f"\n📅 Admission Period:")
        print(f"   Admit Time:    {adm['admittime']}")
        print(f"   Discharge Time: {adm['dischtime']}")
        print(f"   Duration:      {adm['dischtime'] - adm['admittime']}")
        
        if 'admission_type' in adm:
            print(f"   Type:          {adm['admission_type']}")
        if 'admission_location' in adm:
            print(f"   From:          {adm['admission_location']}")
        if 'discharge_location' in adm:
            print(f"   To:            {adm['discharge_location']}")
        
        # EHR Data - Diagnoses
        patient_diag = ehr_diagnoses[
            (ehr_diagnoses['subject_id'] == subject_id) & 
            (ehr_diagnoses['hadm_id'] == adm['hadm_id'])
        ].sort_values('seq_num')
        
        print(f"\n🏥 EHR Data:")
        print(f"   Diagnoses ({len(patient_diag)} total):")
        for _, diag in patient_diag.head(10).iterrows():  # Show first 10
            icd_version = diag.get('icd_version', 'N/A')
            icd_code = diag.get('icd_code', 'N/A')
            seq = diag.get('seq_num', 'N/A')
            print(f"     [{seq}] ICD-{icd_version}: {icd_code}")
        if len(patient_diag) > 10:
            print(f"     ... and {len(patient_diag) - 10} more diagnoses")
        
        # CXR Data
        print(f"\n📷 CXR Data:")
        if len(matching_cxr) > 0:
            print(f"   Studies: {matching_cxr['study_id'].nunique()}, Total Images: {len(matching_cxr)}")
            for study_id in matching_cxr['study_id'].unique():
                study_cxr = matching_cxr[matching_cxr['study_id'] == study_id].sort_values('study_datetime')
                print(f"\n   Study ID: {study_id}")
                for _, cxr in study_cxr.iterrows():
                    print(f"     [{cxr['study_datetime']}] DICOM: {cxr['dicom_id'][:40]}...")
                    print(f"         View: {cxr.get('ViewPosition', 'N/A')}")
        else:
            print(f"   No CXR studies during this admission")
        
        # ECG Data
        print(f"\n📊 ECG Data:")
        if len(matching_ecg) > 0:
            print(f"   Records: {len(matching_ecg)}")
            for _, ecg in matching_ecg.iterrows():
                print(f"\n   [{ecg['ecg_time']}] Study ID: {ecg['study_id']}, Cart: {ecg['cart_id']}")
                # Combine report
                report_parts = []
                for i in range(18):
                    val = ecg[f'report_{i}']
                    if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                        report_parts.append(str(val).strip())
                if report_parts:
                    print(f"     Report: {', '.join(report_parts)}")
                if pd.notna(ecg.get('rr_interval')):
                    print(f"     RR Interval: {ecg['rr_interval']} ms")
        else:
            print(f"   No ECG records during this admission")
        
        print()
    
    print("=" * 80)
    print("END OF PATIENT RECORD")
    print("=" * 80)
else:
    print("\n⚠️  No patient found with all three modalities")


PATIENT DATA IN TIME/HADM_ID SEQUENTIAL FORMAT


NameError: name 'ehr_admissions' is not defined

## MIMIC-IV Notes Dataset Analysis

Examining discharge notes and radiology notes, and their relationships to EHR admissions and CXR studies.


In [ ]:
# Analyze MIMIC-IV Notes dataset
print("=" * 70)
print("MIMIC-IV Notes Dataset Structure")
print("=" * 70)

# Load notes if not already loaded
if 'discharge_notes' not in globals() or 'radiology_notes' not in globals():
    discharge_notes, radiology_notes, _, _ = load_mimiciv_notes(lazy=True)
    discharge_notes['charttime'] = pd.to_datetime(discharge_notes['charttime'])
    discharge_notes['storetime'] = pd.to_datetime(discharge_notes['storetime'])
    radiology_notes['charttime'] = pd.to_datetime(radiology_notes['charttime'])
    radiology_notes['storetime'] = pd.to_datetime(radiology_notes['storetime'])

print(f"\nAvailable files:")
print(f"  1. discharge.csv - {len(discharge_notes)} records")
print(f"  2. radiology.csv - {len(radiology_notes)} records")

print(f"\n" + "=" * 70)
print("Discharge Notes Structure")
print("=" * 70)
print(f"\nColumns: {list(discharge_notes.columns)}")
print(f"\nSample record:")
print(discharge_notes.head(1))

print(f"\n" + "=" * 70)
print("Radiology Notes Structure")
print("=" * 70)
print(f"\nColumns: {list(radiology_notes.columns)}")
print(f"\nSample record:")
print(radiology_notes.head(1))


In [ ]:
# Check Discharge Notes ↔ EHR Admissions Relationship
print("=" * 70)
print("Discharge Notes ↔ EHR Admissions Relationship")
print("=" * 70)

print(f"\n✅ Discharge notes have hadm_id column!")

print(f"\nStatistics:")
print(f"  Total discharge notes: {len(discharge_notes)}")
print(f"  Unique hadm_ids in discharge: {discharge_notes['hadm_id'].nunique()}")
print(f"  Unique hadm_ids in admissions: {ehr_admissions['hadm_id'].nunique()}")

# Check one-to-one relationship
hadm_counts = discharge_notes.groupby('hadm_id').size()
print(f"\n  Discharge notes per hadm_id:")
print(f"    Min: {hadm_counts.min()}, Max: {hadm_counts.max()}, Mean: {hadm_counts.mean():.2f}")
print(f"    Hadm_ids with multiple notes: {(hadm_counts > 1).sum()}")

print(f"\n✅ CONCLUSION: Each discharge note corresponds to exactly one EHR admission (1:1 relationship)")
print(f"   - Each hadm_id has exactly 1 discharge note")
print(f"   - Can directly link discharge notes to admissions via hadm_id")


In [ ]:
# Check Radiology Notes ↔ CXR Studies Relationship
print("=" * 70)
print("Radiology Notes ↔ CXR Studies Relationship")
print("=" * 70)

print(f"\n⚠️  Radiology notes do NOT have study_id column directly")
print(f"   They have: subject_id, hadm_id, charttime, storetime")

# Check if we can match by text content (looking for study_id mentions)
print(f"\nChecking if radiology notes mention study_id in text...")

# Sample radiology note
sample_rad = radiology_notes.iloc[0]
print(f"\nSample radiology note:")
print(f"  Note ID: {sample_rad['note_id']}")
print(f"  Subject ID: {sample_rad['subject_id']}")
print(f"  HADM ID: {sample_rad['hadm_id']}")
print(f"  Chart Time: {sample_rad['charttime']}")
print(f"  Store Time: {sample_rad['storetime']}")
print(f"\n  Text preview (first 300 chars):")
print(f"  {sample_rad['text'][:300]}...")

# Try temporal matching
print(f"\n" + "=" * 70)
print("Temporal Matching: Radiology Notes ↔ CXR Studies")
print("=" * 70)

# Find a patient with both radiology notes and CXR
sample_subject = radiology_notes['subject_id'].iloc[0]
patient_rad = radiology_notes[radiology_notes['subject_id'] == sample_subject].sort_values('charttime')
patient_cxr = cxr_metadata[cxr_metadata['subject_id'] == sample_subject].sort_values('study_datetime')

# Filter for chest-related radiology notes
chest_rad = patient_rad[patient_rad['text'].str.contains('CHEST|CXR|CHEST X', case=False, na=False)]

print(f"\nPatient {sample_subject}:")
print(f"  Total radiology notes: {len(patient_rad)}")
print(f"  Chest-related notes: {len(chest_rad)}")
print(f"  CXR studies: {patient_cxr['study_id'].nunique()}")

# Try matching by time window
print(f"\n  Attempting time-based matching (within 24 hours):")
matches_found = 0
for _, rad in chest_rad.head(5).iterrows():
    rad_time = rad['charttime'] if pd.notna(rad['charttime']) else rad['storetime']
    if pd.notna(rad_time):
        nearby_cxr = patient_cxr[
            (patient_cxr['study_datetime'] >= rad_time - pd.Timedelta(hours=24)) &
            (patient_cxr['study_datetime'] <= rad_time + pd.Timedelta(hours=24))
        ]
        if len(nearby_cxr) > 0:
            matches_found += 1
            print(f"\n    ✅ Match found:")
            print(f"       Radiology note: {rad['note_id']} at {rad_time}")
            print(f"       Matched CXR studies: {nearby_cxr['study_id'].nunique()}")
            for study_id in nearby_cxr['study_id'].unique():
                study_cxr = nearby_cxr[nearby_cxr['study_id'] == study_id]
                print(f"         Study {study_id} at {study_cxr['study_datetime'].iloc[0]}")

print(f"\n✅ CONCLUSION: Radiology notes can be matched to CXR studies via:")
print(f"   1. Same subject_id")
print(f"   2. Temporal matching (charttime/storetime vs StudyDate/StudyTime)")
print(f"   3. Text content filtering (CHEST, CXR keywords)")


## Summary: Patient Record Matching Logic

Complete overview of how to match and unify patient records across all modalities.


In [ ]:
# Comprehensive summary of patient record matching logic
print("=" * 80)
print("PATIENT RECORD MATCHING LOGIC - COMPREHENSIVE SUMMARY")
print("=" * 80)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                           OVERALL STRUCTURE                                 │
└─────────────────────────────────────────────────────────────────────────────┘

Patient (subject_id) - The root identifier that links all modalities
    │
    └── Admission (hadm_id) - Hospital visit/stay, defined by time window
            │
            ├── EHR Data (linked via hadm_id)
            ├── Discharge Notes (linked via hadm_id)
            ├── CXR Studies (matched by time)
            ├── Radiology Notes (matched by time + content)
            └── ECG Records (matched by time)
""")

print("\n" + "=" * 80)
print("STEP-BY-STEP MATCHING LOGIC")
print("=" * 80)

print("""
STEP 1: Start with EHR Admissions
  ┌─────────────────────────────────────────────────────────────────┐
  │  • Load admissions.csv                                         │
  │  • Each row = one admission (hadm_id)                          │
  │  • Has: subject_id, hadm_id, admittime, dischtime              │
  │  • Time window: admittime to dischtime defines the admission   │
  └─────────────────────────────────────────────────────────────────┘

STEP 2: Link EHR Data to Admissions (Direct Link)
  ┌─────────────────────────────────────────────────────────────────┐
  │  • All EHR data has hadm_id column                             │
  │  • Direct link via hadm_id - no temporal matching needed      │
  │                                                                 │
  │  Data linked via hadm_id:                                      │
  │    ✓ diagnoses_icd.csv      → ICD diagnosis codes             │
  │    ✓ procedures_icd.csv     → ICD procedure codes             │
  │    ✓ labevents.csv          → Laboratory results              │
  │    ✓ prescriptions.csv      → Medications                     │
  │    ✓ And other clinical tables                                │
  └─────────────────────────────────────────────────────────────────┘

STEP 3: Link Discharge Notes (Direct Link)
  ┌─────────────────────────────────────────────────────────────────┐
  │  • discharge.csv has hadm_id column                            │
  │  • 1:1 relationship: each hadm_id = exactly 1 discharge note  │
  │  • Direct link: discharge_note.hadm_id == admission.hadm_id     │
  │  • No temporal matching needed                                  │
  └─────────────────────────────────────────────────────────────────┘

STEP 4: Match CXR Studies to Admissions (Temporal Matching)
  ┌─────────────────────────────────────────────────────────────────┐
  │  • CXR metadata has: subject_id, study_id, StudyDate, StudyTime│
  │  • CXR does NOT have hadm_id                                    │
  │                                                                 │
  │  Matching Rule:                                                │
  │    IF subject_id matches AND                                 │
  │       admittime <= CXR_study_datetime <= dischtime             │
  │    THEN CXR study belongs to this admission                   │
  │                                                                 │
  │  Where:                                                        │
  │    CXR_study_datetime = StudyDate + StudyTime                 │
  │                                                                 │
  │  Note:                                                         │
  │    • One study_id can have multiple images (dicom_ids)        │
  │    • Multiple CXR studies can belong to one admission         │
  └─────────────────────────────────────────────────────────────────┘

STEP 5: Match Radiology Notes to CXR Studies (Temporal + Content Matching)
  ┌─────────────────────────────────────────────────────────────────┐
  │  • radiology.csv has: subject_id, hadm_id, charttime, storetime│
  │  • Radiology notes do NOT have study_id                        │
  │                                                                 │
  │  Matching Strategy:                                            │
  │    1. Filter for chest-related notes (text contains CHEST/CXR)│
  │    2. Match by subject_id (same patient)                       │
  │    3. Temporal matching:                                      │
  │       radiology_time ≈ CXR_study_datetime (within 24 hours)   │
  │                                                                 │
  │  Where:                                                        │
  │    radiology_time = charttime (or storetime if charttime N/A) │
  │                                                                 │
  │  Note:                                                         │
  │    • First match radiology to CXR, then link to admission     │
  │    • One radiology note typically matches one CXR study        │
  └─────────────────────────────────────────────────────────────────┘

STEP 6: Match ECG Records to Admissions (Temporal Matching)
  ┌─────────────────────────────────────────────────────────────────┐
  │  • ECG has: subject_id, study_id, ecg_time                     │
  │  • ECG does NOT have hadm_id                                    │
  │                                                                 │
  │  Matching Rule:                                                │
  │    IF subject_id matches AND                                 │
  │       admittime <= ecg_time <= dischtime                      │
  │    THEN ECG record belongs to this admission                  │
  │                                                                 │
  │  Note:                                                         │
  │    • ECG study_id is unique per record (1:1)                 │
  │    • Multiple ECG records can belong to one admission         │
  └─────────────────────────────────────────────────────────────────┘
""")

print("\n" + "=" * 80)
print("KEY IDENTIFIERS SUMMARY")
print("=" * 80)

print("""
Common Identifier (Links All Modalities):
  • subject_id - Patient identifier, shared across ALL datasets

EHR-Specific Identifiers:
  • hadm_id - Hospital admission ID (primary grouping unit)
  • Used in: admissions, diagnoses, procedures, labs, prescriptions, etc.

CXR-Specific Identifiers:
  • study_id - Study ID (unique to CXR, one study can have multiple images)
  • dicom_id - Individual image ID
  • study_datetime - Combined from StudyDate + StudyTime

ECG-Specific Identifiers:
  • study_id - Study ID (unique to ECG, 1:1 with records)
  • cart_id - Device identifier
  • ecg_time - Temporal identifier

Notes-Specific Identifiers:
  • note_id - Unique note identifier
  • hadm_id - For discharge notes (direct link to admissions)
  • charttime/storetime - Temporal identifiers for radiology notes
""")

print("\n" + "=" * 80)
print("MATCHING STRATEGY SUMMARY TABLE")
print("=" * 80)

print("""
┌─────────────────────┬──────────────┬──────────────────┬────────────────────┐
│   Data Type         │ Linking Key  │ Matching Method  │ Notes              │
├─────────────────────┼──────────────┼──────────────────┼────────────────────┤
│ EHR Data            │ hadm_id      │ Direct Link      │ All EHR tables    │
│                     │              │                  │ have hadm_id      │
├─────────────────────┼──────────────┼──────────────────┼────────────────────┤
│ Discharge Notes     │ hadm_id      │ Direct Link      │ 1:1 relationship  │
├─────────────────────┼──────────────┼──────────────────┼────────────────────┤
│ CXR Studies         │ subject_id   │ Temporal Match   │ Time window match │
│                     │ + time       │                  │ to admission       │
├─────────────────────┼──────────────┼──────────────────┼────────────────────┤
│ Radiology Notes     │ subject_id   │ Temporal +       │ Match to CXR first│
│                     │ + time       │ Content Match    │ then to admission │
├─────────────────────┼──────────────┼──────────────────┼────────────────────┤
│ ECG Records         │ subject_id   │ Temporal Match  │ Time window match │
│                     │ + time       │                  │ to admission       │
└─────────────────────┴──────────────┴──────────────────┴────────────────────┘
""")

print("\n" + "=" * 80)
print("IMPLEMENTATION APPROACH")
print("=" * 80)

print("""
1. Start with EHR Admissions (hadm_id as primary grouping unit)
   → This provides the time windows and structure

2. For each admission:
   a. Load EHR data via hadm_id (direct link)
   b. Load discharge note via hadm_id (direct link)
   c. Find CXR studies: subject_id match + time in window
   d. Find ECG records: subject_id match + time in window
   e. Find radiology notes: 
      - Filter by subject_id + hadm_id (if available)
      - Or match via temporal proximity to CXR studies

3. Result: Unified patient record structure
   Patient (subject_id)
     └── Admission (hadm_id, admittime, dischtime)
         ├── EHR: diagnoses, procedures, labs, prescriptions
         ├── Discharge Note: text
         ├── CXR: studies with images
         │   └── Radiology Notes: matched to CXR studies
         └── ECG: records with reports

4. Handle unmatched records:
   • CXR/ECG outside admission windows → store as unmatched
   • Can create "virtual admissions" if needed for unmatched data
""")

print("\n" + "=" * 80)
print("END OF SUMMARY")
print("=" * 80)
